## Reading the raw stores data

In [0]:
data = spark.read.table("`01_bronze`.raw.stores")
display(data)

## Date Normalization to standard format

In [0]:
import pyspark.sql.functions as F

def standardize_date(columnName, df):
    return df.withColumn(
        "date_parsed",
        F.coalesce(
            F.try_to_date(F.col(columnName), "M/d/yyyy"),
            F.try_to_date(F.col(columnName), "M-d-yyyy"),
            F.try_to_date(F.col(columnName), "yyyy-M-d"),
            F.try_to_date(F.col(columnName), "yyyy/M/d")
        )
    ).withColumn(
        columnName,
        F.trim(F.col("date_parsed")).cast('date')
    ).drop("date_parsed")

data = standardize_date("open_date", data)
display(data)


## Data type's change

In [0]:
def change_dataType(df,dataType,column):
    df = df.withColumn(column,df[column].cast(dataType))
    return df
data = change_dataType(data,"int","storekey")
data = change_dataType(data,"int","square_meters")
display(data)

## write table to delta lake 

In [0]:
data.write.mode("overwrite").saveAsTable("`02_silver`.transformation.stores")